# 13 — BERTScore (backfill sulla split test)

Aggiunge il **BERTScore** (Zhang et al., 2020) alle metriche gia' calcolate per ciascun metodo:
similarita' semantica tra riassunto generato e riferimento basata su embedding contestuali BERT
(cosine similarity token-per-token, poi aggregata in Precision/Recall/F1), invece della sola
sovrapposizione lessicale di ROUGE/BLEU/METEOR.

## Perche' non `psr.bert_score()`

`pyAutoSummarizer` (gia' usato per ROUGE/BLEU/METEOR, vedi `summ_utils.crea_valutatore`)
espone anche un wrapper di comodo `bert_score()`. Ispezionando il suo codice
(`pyAutoSummarizer/base/psr.py`) e quello della libreria che chiama, `bert_score.score()`
(`bert_score/utils.py:get_model`), risulta che **ogni chiamata ricarica `roberta-large` da
zero** (nessuna cache tra chiamate). Usarlo dentro il ciclo per-esempio esistente
(`metriche_esempio`, una chiamata per riga) vorrebbe dire ricaricare un modello da ~355M
parametri 5.600+ volte per metodo — impraticabile (stimate 40-60+ ore in totale sui 13
metodi).

Questo notebook usa invece `bert_score.BERTScorer` **direttamente**: il modello viene
caricato **una sola volta** e tutte le coppie candidato/riferimento di un metodo vengono
valutate in un'**unica chiamata batched** (`calcola_bertscore_batch` in `summ_utils.py`).
Stima (misurata su questa macchina, GPU RTX PRO 2000 Blackwell Laptop, su un sottoinsieme di
300 righe di `firstk_psr`): caricamento del modello **~6 s** (una tantum per metodo) e
**~21 righe/s** di scoring puro — circa **~1 ora di calcolo GPU** per 13 metodi sulla split
test (~5.600 righe ciascuno), quindi ~25 minuti per i 5 metodi del backfill successivo.

## Ambito e portata

**Backfill** sulla sola split `test` (5.610 righe, gia' generate da tutti i
metodi — vedi `results/summaries/`): non modifica i notebook 01-04/06-12/15-17 ne' il loro
ciclo di generazione/valutazione dal vivo. Il notebook 05 (confronto) legge
automaticamente le nuove colonne `bertscore_f1/p/r` una volta che questo notebook e' stato
eseguito.

I 18 metodi sono gli stessi della Vista 2 del notebook 05. Il notebook e' **incrementale**:
un metodo che ha gia' le colonne `bertscore_*` nel suo CSV per-esempio viene saltato, perche'
`valuta_e_salva` riscrive CSV e JSON del metodo e ricalcolarli rigenererebbe file gia'
pubblicati a parita' di valori. E' cosi' che il backfill dei cinque metodi dei notebook 15-17
(issue #12) e' girato sulla lista completa senza toccare i 13 preesistenti; per un ricalcolo
integrale si imposta `BERTSCORE_FORZA=1`.

TextRank/LexRank non hanno una
corsa `_test.tsv` dedicata (solo `_full.tsv`, l'intero `complete.tab`): qui le loro
metriche ROUGE/BLEU/METEOR vengono **ricalcolate direttamente** sulle sole righe della
split test (invece di essere derivate filtrando la corsa `full`, come fa
`scripts/run_benchmark_test.py`) — numericamente equivalente, dato che le metriche sono
calcolate per esempio, ma piu' semplice da tenere in un unico ciclo uniforme su tutti i
metodi. La configurazione storica di ciascun metodo (`config` nel JSON aggregato, se
gia' presente) viene preservata cosi' com'e', non sovrascritta da questo backfill.


In [1]:
# Installa le dipendenze se mancanti (per esempio su Google Colab)
try:
    import pyAutoSummarizer  # noqa: F401  (ROUGE/BLEU/METEOR, invariato rispetto agli altri notebook)
except ImportError:
    %pip install pyAutoSummarizer sentencepiece

try:
    import bert_score  # noqa: F401
except ImportError:
    %pip install bert-score


C:\Users\antonio.girasella\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- Configurazione ---------------------------------------------------------
import csv
import json
import os
import time

import summ_utils as su

SCOPE      = 'test'           # backfill una tantum sulla split test (vedi notebook 05, Vista 2)
MODEL_TYPE = 'roberta-large'  # modello di default di bert-score per l'inglese
BATCH_SIZE = 64
DEVICE     = su.rileva_device()

BASE = su.trova_base_dir()
P    = su.percorsi_standard(BASE)

# Stessi 18 metodi con metriche 'test' del notebook 05 (Vista 2)
METODI_BASELINE = ['firstk_psr', 'firstk_nltk',          # notebook 10 (First-k)
                   'centroid_mmr', 'centroid_mmr_bert']  # notebook 11 (Centroid+MMR)
METODI_NON_SUP = ['lsa', 'lsa_steinberger',        # notebook 15 (LSA/SVD, due selezioni)
                  'sbert_kmeans', 'sbert_agglom',  # notebook 16 (clustering su embedding SBERT)
                  'lda']                           # notebook 17 (topic modeling LDA)
METODI = METODI_BASELINE + ['textrank', 'lexrank', 'bart', 'pegasus', 'primera',
                            'qwen', 'gemma', 'mistral'] + ['gpt5mini'] + METODI_NON_SUP

# I metodi che hanno gia' le colonne BERTScore vengono saltati: `valuta_e_salva`
# RISCRIVE il CSV per-esempio e il JSON aggregato del metodo, quindi ricalcolarli
# rigenererebbe file gia' pubblicati senza cambiarne i valori. E' cosi' che il
# backfill dei cinque metodi dei notebook 15-17 (issue #12) ha potuto girare sulla
# lista completa senza toccare i 13 preesistenti. BERTSCORE_FORZA=1 ricalcola tutto.
FORZA_RICALCOLO = os.environ.get('BERTSCORE_FORZA') == '1'


def percorso_riassunti(metodo):
    """TextRank/LexRank non hanno una corsa '_test.tsv' dedicata: i riassunti sono
    nel file '_full.tsv' (intero complete.tab). I riferimenti sotto sono comunque gia'
    ristretti alla sola split test, quindi il filtro avviene automaticamente."""
    suffisso = 'full' if metodo in ('textrank', 'lexrank') else 'test'
    return P['summaries_dir'] / f'{metodo}_{suffisso}.tsv'


def bertscore_gia_presente(metodo):
    """True se il CSV per-esempio del metodo ha gia' le colonne BERTScore."""
    csv_path = P['metrics_dir'] / f'{metodo}_{SCOPE}_per_example.csv'
    if not csv_path.exists():
        return False
    with open(csv_path, encoding='utf-8', newline='') as f:
        intestazione = next(csv.reader(f), [])
    return all(c in intestazione for c in su.COLONNE_METRICHE_BERTSCORE)


da_calcolare = [m for m in METODI if FORZA_RICALCOLO or not bertscore_gia_presente(m)]

print(f'Modello BERTScore : {MODEL_TYPE}')
print(f'Device            : {DEVICE}')
print(f'Metodi            : {len(METODI)} totali')
print(f'Da calcolare      : {da_calcolare or "nessuno (tutti gia\' con BERTScore)"}')


Modello BERTScore : roberta-large
Device            : cuda
Metodi            : 18 totali
Da calcolare      : ['lsa', 'lsa_steinberger', 'sbert_kmeans', 'sbert_agglom', 'lda']


## Backfill

Riferimenti della split test caricati **una sola volta** (streaming su `complete.tab`,
poi tenuti in memoria: ~5.610 righe) e riusati per tutti i metodi, invece di
rileggere il file da 658 MB a ogni iterazione.


In [3]:
riferimenti_test = list(su.itera_split(P['complete_tab'], 'test'))
print(f'Riferimenti split test: {len(riferimenti_test)} righe')


Riferimenti split test: 5610 righe


In [4]:
risultati_overall = {}
for metodo in METODI:
    if metodo not in da_calcolare:
        # Gia' fatto in una corsa precedente: si rilegge il valore dal JSON aggregato
        # per il riepilogo finale, senza riscrivere nulla.
        json_esistente = P['metrics_dir'] / f'{metodo}_{SCOPE}_aggregate.json'
        with open(json_esistente, encoding='utf-8') as f:
            risultati_overall[metodo] = json.load(f)['overall']
        print(f'({metodo}: BERTScore gia\' presente, salto)')
        continue

    riassunti_path = percorso_riassunti(metodo)
    if not riassunti_path.exists():
        print(f'({metodo}: {riassunti_path.name} non trovato, salto)')
        continue
    riassunti = su.carica_riassunti(riassunti_path)

    coppie = [(rif['row_id'], riassunti[rif['row_id']], su.pulisci_riferimento(rif['summary']))
              for rif in riferimenti_test if rif['row_id'] in riassunti]
    if not coppie:
        print(f'({metodo}: nessuna riga in comune con la split test, salto)')
        continue

    t0 = time.time()
    extra = su.calcola_bertscore_batch(coppie, model_type=MODEL_TYPE, device=DEVICE,
                                       batch_size=BATCH_SIZE)
    durata = time.time() - t0
    print(f'{metodo}: BERTScore su {len(coppie)} righe in {durata:.0f}s '
          f'({len(coppie) / durata:.1f} righe/s)')

    # Config storica del metodo, se gia' presente: preservata cosi' com'e', non
    # sovrascritta da questo backfill (contiene i parametri della corsa originale).
    json_esistente = P['metrics_dir'] / f'{metodo}_test_aggregate.json'
    config = {}
    if json_esistente.exists():
        with open(json_esistente, encoding='utf-8') as f:
            config = json.load(f).get('config', {})
    if metodo in ('textrank', 'lexrank'):
        config = {**config, 'nota_bertscore':
                  "rouge/bleu/meteor ricalcolati direttamente sulle sole righe della split "
                  "test (non piu' filtrando la corsa full) in occasione del backfill "
                  "BERTScore (notebook 13); valori numericamente equivalenti alla "
                  "derivazione precedente (scripts/run_benchmark_test.py)."}

    righe, aggregato = su.valuta_e_salva(riferimenti_test, riassunti, metodo, SCOPE,
                                         P['metrics_dir'], config,
                                         extra_metriche=extra,
                                         colonne_extra=su.COLONNE_METRICHE_BERTSCORE)
    risultati_overall[metodo] = aggregato['overall']

print('\nBERTScore F1 medio per metodo:')
print(json.dumps({m: round(v.get('bertscore_f1', float('nan')), 4)
                  for m, v in risultati_overall.items()}, indent=2))


(firstk_psr: BERTScore gia' presente, salto)
(firstk_nltk: BERTScore gia' presente, salto)
(centroid_mmr: BERTScore gia' presente, salto)
(centroid_mmr_bert: BERTScore gia' presente, salto)
(textrank: BERTScore gia' presente, salto)
(lexrank: BERTScore gia' presente, salto)
(bart: BERTScore gia' presente, salto)
(pegasus: BERTScore gia' presente, salto)
(primera: BERTScore gia' presente, salto)
(qwen: BERTScore gia' presente, salto)
(gemma: BERTScore gia' presente, salto)
(mistral: BERTScore gia' presente, salto)
(gpt5mini: BERTScore gia' presente, salto)


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 5023.30it/s]


[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


lsa: BERTScore su 5588 righe in 287s (19.5 righe/s)


...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 6182.73it/s]


[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


lda: BERTScore su 5588 righe in 329s (17.0 righe/s)


Metriche per-esempio : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\lda_test_per_example.csv (5588 righe)
Metriche aggregate   : C:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\lda_test_aggregate.json

BERTScore F1 medio per metodo:
{
  "firstk_psr": 0.8396,
  "firstk_nltk": 0.8426,
  "centroid_mmr": 0.8416,
  "centroid_mmr_bert": 0.8409,
  "textrank": 0.8405,
  "lexrank": 0.837,
  "bart": 0.8541,
  "pegasus": 0.8711,
  "primera": 0.8742,
  "qwen": 0.8568,
  "gemma": 0.839,
  "mistral": 0.8598,
  "gpt5mini": 0.8523,
  "lsa": 0.8329,
  "lsa_steinberger": 0.8421,
  "sbert_kmeans": 0.8396,
  "sbert_agglom": 0.8325,
  "lda": 0.8347
}


## Validazione consigliata prima della corsa completa

Prima di eseguire questo notebook su molti metodi, e' consigliabile validare la
stima di tempo su un solo metodo veloce (es. `firstk_psr`, ~5.588 righe): impostare
temporaneamente `METODI = ['firstk_psr']` nella cella di configurazione con
`BERTSCORE_FORZA=1`, eseguire, e confrontare le righe/s misurate con la stima riportata
sopra prima di ripristinare la lista completa. Dopo la corsa, ri-eseguire il notebook 05
per aggiornare le viste di confronto con la nuova colonna BERTScore.
